In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,LSTM
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [6]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [7]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [8]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [10]:
SEQ_LEN = 24
HORIZON = 24

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101593, 24, 13)
X_test Shape:  (21733, 24, 13)


In [11]:
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

In [12]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [13]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [ ]:
#dropout 0.2

In [14]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0071 - mae: 0.0581 - val_loss: 0.0026 - val_mae: 0.0381
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0024 - mae: 0.0374 - val_loss: 0.0021 - val_mae: 0.0338
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0018 - val_mae: 0.0309
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0018 - mae: 0.0320 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0018 - mae: 0.0312 - val_loss: 0.0019 - val_mae: 0.0310
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0016 - val_mae: 0.0281
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [15]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1388.15 MW
RMSE: 1958.44 MW
MAPE: 4.38%
R2: 0.91%


In [16]:
model_lstm.save(r'../models/early_rnn.keras')
history_df =pd.DataFrame(history_lstm.history)

history_df.to_csv(r'../log/early_rnn.csv',index=False)

In [ ]:
#0.3

In [12]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0073 - mae: 0.0612 - val_loss: 0.0027 - val_mae: 0.0392
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0027 - mae: 0.0396 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0022 - mae: 0.0356 - val_loss: 0.0018 - val_mae: 0.0318
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0339 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0019 - mae: 0.0323 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0018 - mae: 0.0320 - val_loss: 0.0017 - val_mae: 0.0295
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [13]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1455.97 MW
RMSE: 1999.42 MW
MAPE: 4.67%
R2: 0.90%


In [ ]:
#0.5

In [17]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0089 - mae: 0.0665 - val_loss: 0.0028 - val_mae: 0.0410
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0033 - mae: 0.0440 - val_loss: 0.0025 - val_mae: 0.0384
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0027 - mae: 0.0391 - val_loss: 0.0020 - val_mae: 0.0323
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0025 - mae: 0.0377 - val_loss: 0.0019 - val_mae: 0.0318
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0024 - mae: 0.0368 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0023 - mae: 0.0360 - val_loss: 0.0017 - val_mae: 0.0303
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0018 - val_mae: 0.0308
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [18]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1537.21 MW
RMSE: 2104.62 MW
MAPE: 4.84%
R2: 0.89%


In [ ]:
#batch nornmalization

In [14]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

E0000 00:00:1786423889.073521   50319 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0385 - mae: 0.1191 - val_loss: 0.0045 - val_mae: 0.0538
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0057 - mae: 0.0593 - val_loss: 0.0039 - val_mae: 0.0483
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0039 - mae: 0.0484 - val_loss: 0.0030 - val_mae: 0.0426
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0031 - mae: 0.0431 - val_loss: 0.0024 - val_mae: 0.0375
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0027 - mae: 0.0401 - val_loss: 0.0022 - val_mae: 0.0346
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0025 - mae: 0.0379 - val_loss: 0.0021 - val_mae: 0.0343
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0023 - mae: 0.0365 - val_loss: 0.0018 - val_mae: 0.0313
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0022 - mae: 0.0355 - val_loss: 0.0018 - val_mae: 0.0314
Epoch 9/10
1588/1588 ━━━━

In [15]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1459.67 MW
RMSE: 2010.76 MW
MAPE: 4.63%
R2: 0.90%


In [ ]:
#optimizer

In [16]:
model_lstm = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0047 - mae: 0.0473 - val_loss: 0.0022 - val_mae: 0.0347
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0018 - mae: 0.0312 - val_loss: 0.0019 - val_mae: 0.0319
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0016 - mae: 0.0291 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0015 - mae: 0.0281 - val_loss: 0.0017 - val_mae: 0.0307
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0014 - mae: 0.0273 - val_loss: 0.0016 - val_mae: 0.0288
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0014 - mae: 0.0267 - val_loss: 0.0016 - val_mae: 0.0289
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0013 - mae: 0.0263 - val_loss: 0.0015 - val_mae: 0.0275
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0013 - mae: 0.0259 - val_loss: 0.0015 - val_mae: 0.0281
Epoch 9/10
1588/1588 ━━━━━

In [17]:
y_pred_lstm_scaled = model_lstm.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1395.00 MW
RMSE: 1957.30 MW
MAPE: 4.40%
R2: 0.91%


In [18]:
model_lstm.save(r'../models/lstm_adam.keras')
history_df =pd.DataFrame(history_lstm.history)

history_df.to_csv(r'../log/lstm_adam.csv',index=False)

In [ ]:
#sgd

In [19]:
model_lstm_sgd = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_sgd.compile(optimizer='sgd', loss='mse', metrics=['mae'])

history_lstm = model_lstm_sgd.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0319 - mae: 0.1334 - val_loss: 0.0181 - val_mae: 0.1100
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0158 - mae: 0.0982 - val_loss: 0.0154 - val_mae: 0.1007
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0140 - mae: 0.0925 - val_loss: 0.0134 - val_mae: 0.0929
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0127 - mae: 0.0881 - val_loss: 0.0118 - val_mae: 0.0868
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0117 - mae: 0.0850 - val_loss: 0.0108 - val_mae: 0.0827
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0110 - mae: 0.0827 - val_loss: 0.0101 - val_mae: 0.0801
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0105 - mae: 0.0810 - val_loss: 0.0097 - val_mae: 0.0781
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0101 - mae: 0.0796 - val_loss: 0.0093 - val_mae: 0.0764
Epoch 9/10
1588/1588 ━━━━━━

In [20]:
y_pred_lstm_scaled = model_lstm_sgd.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  3577.38 MW
RMSE: 4564.26 MW
MAPE: 11.68%
R2: 0.50%


In [ ]:
#rms prop

In [21]:
model_lstm_rms = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_rms.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='mse', metrics=['mae'])

history_lstm = model_lstm_rms.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0057 - mae: 0.0543 - val_loss: 0.0035 - val_mae: 0.0462
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0025 - mae: 0.0376 - val_loss: 0.0025 - val_mae: 0.0380
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0020 - mae: 0.0336 - val_loss: 0.0021 - val_mae: 0.0336
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0018 - mae: 0.0318 - val_loss: 0.0021 - val_mae: 0.0345
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0021 - val_mae: 0.0333
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0017 - mae: 0.0301 - val_loss: 0.0022 - val_mae: 0.0360
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0018 - val_mae: 0.0305
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0016 - mae: 0.0291 - val_loss: 0.0022 - val_mae: 0.0337
Epoch 9/10
1588/1588 ━━━━

In [22]:
y_pred_lstm_scaled = model_lstm_rms.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1493.50 MW
RMSE: 2080.44 MW
MAPE: 4.71%
R2: 0.90%


In [ ]:
#early stopping

In [25]:
model_lstm_early = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_early.compile(optimizer=tf.keras.optimizers.Adam(), loss='mse', metrics=['mae'])

history_lstm = model_lstm_early.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,callbacks=[early_stop])

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0051 - mae: 0.0479 - val_loss: 0.0021 - val_mae: 0.0334
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0019 - val_mae: 0.0318
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0016 - mae: 0.0287 - val_loss: 0.0017 - val_mae: 0.0297
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0015 - mae: 0.0278 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0014 - mae: 0.0270 - val_loss: 0.0016 - val_mae: 0.0291
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0014 - mae: 0.0265 - val_loss: 0.0016 - val_mae: 0.0283
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0013 - mae: 0.0259 - val_loss: 0.0015 - val_mae: 0.0280
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0013 - mae: 0.0255 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 9/10
1588/1588 ━━━━

In [26]:
y_pred_lstm_scaled = model_lstm_early.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1364.05 MW
RMSE: 1927.57 MW
MAPE: 4.28%
R2: 0.91%


In [27]:
model_lstm.save(r'../models/lstm_early.keras')
history_df =pd.DataFrame(history_lstm.history)

history_df.to_csv(r'../log/lstm_early.csv',index=False)

In [ ]:
#learinig rate

In [28]:
model_lstm_lr = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_lr.compile(optimizer=tf.keras.optimizers.Adam(), loss='mse', metrics=['mae'])

history_lstm = model_lstm_lr.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,callbacks=[lr_scheduler])

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.0057 - mae: 0.0500 - val_loss: 0.0025 - val_mae: 0.0381 - learning_rate: 0.0010
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0019 - mae: 0.0319 - val_loss: 0.0019 - val_mae: 0.0315 - learning_rate: 0.0010
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0019 - val_mae: 0.0311 - learning_rate: 0.0010
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0015 - mae: 0.0282 - val_loss: 0.0017 - val_mae: 0.0299 - learning_rate: 0.0010
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0014 - mae: 0.0273 - val_loss: 0.0017 - val_mae: 0.0295 - learning_rate: 0.0010
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0014 - mae: 0.0267 - val_loss: 0.0016 - val_mae: 0.0287 - learning_rate: 0.0010
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0013 - mae: 0.0261 - val_loss: 0.0016 - val_mae: 0.0281 - lea

In [29]:
y_pred_lstm_scaled = model_lstm_lr.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1410.04 MW
RMSE: 1971.45 MW
MAPE: 4.42%
R2: 0.91%


In [30]:
model_lstm.save(r'../models/lstm_lr.keras')
history_df =pd.DataFrame(history_lstm.history)

history_df.to_csv(r'../log/lstm_lr.csv',index=False)

In [ ]:
#batch size

In [31]:
model_lstm_32 = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_32.compile(optimizer=tf.keras.optimizers.Adam(), loss='mse', metrics=['mae'])

history_lstm = model_lstm_32.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=32)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0040 - mae: 0.0434 - val_loss: 0.0020 - val_mae: 0.0319
Epoch 2/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0018 - val_mae: 0.0305
Epoch 3/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - loss: 0.0015 - mae: 0.0281 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 4/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0014 - mae: 0.0271 - val_loss: 0.0016 - val_mae: 0.0285
Epoch 5/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0013 - mae: 0.0263 - val_loss: 0.0016 - val_mae: 0.0282
Epoch 6/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0013 - mae: 0.0256 - val_loss: 0.0014 - val_mae: 0.0270
Epoch 7/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0012 - mae: 0.0252 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 8/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - loss: 0.0012 - mae: 0.0248 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 9/10
3175/3175 ━━━━━━━━━━━

In [32]:
y_pred_lstm_scaled = model_lstm_32.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1418.46 MW
RMSE: 1966.94 MW
MAPE: 4.43%
R2: 0.91%


In [ ]:
#16

In [41]:
model_lstm_16 = Sequential([

    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),

    Dense(64, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_32.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm_32.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=5,batch_size=16,callbacks=[early_stop])

Epoch 1/5
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 47s 7ms/step - loss: 9.0679e-04 - mae: 0.0215 - val_loss: 0.0016 - val_mae: 0.0275
Epoch 2/5
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 61s 10ms/step - loss: 8.9080e-04 - mae: 0.0214 - val_loss: 0.0016 - val_mae: 0.0279
Epoch 3/5
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 64s 10ms/step - loss: 8.7397e-04 - mae: 0.0212 - val_loss: 0.0016 - val_mae: 0.0272
Epoch 4/5
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 55s 9ms/step - loss: 8.5921e-04 - mae: 0.0210 - val_loss: 0.0016 - val_mae: 0.0270
Epoch 5/5
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 55s 9ms/step - loss: 8.4392e-04 - mae: 0.0209 - val_loss: 0.0015 - val_mae: 0.0268
Restoring model weights from the end of the best epoch: 5.


In [42]:
y_pred_lstm_scaled = model_lstm_16.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  18670.27 MW
RMSE: 20896.33 MW
MAPE: 58.40%
R2: -9.52%


In [ ]:
#number of layers

In [48]:
model_lstm_layers = Sequential([

    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),

    LSTM(32,return_sequences=False),
    Dropout(0.2),

    Dense(32, activation='relu'),
    
    Dense(HORIZON)
])

model_lstm_layers.compile(optimizer=tf.keras.optimizers.Adam(), loss='mse', metrics=['mae'])

history_lstm = model_lstm_layers.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 35s 20ms/step - loss: 0.0073 - mae: 0.0585 - val_loss: 0.0028 - val_mae: 0.0393
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0026 - mae: 0.0390 - val_loss: 0.0022 - val_mae: 0.0342
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 0.0023 - mae: 0.0358 - val_loss: 0.0019 - val_mae: 0.0318
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 34s 21ms/step - loss: 0.0021 - mae: 0.0346 - val_loss: 0.0019 - val_mae: 0.0326
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 34s 21ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0018 - val_mae: 0.0314
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 43s 27ms/step - loss: 0.0018 - mae: 0.0318 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 34s 21ms/step - loss: 0.0018 - mae: 0.0312 - val_loss: 0.0017 - val_mae: 0.0297
Epoch 9/10
1588/1588 ━━━

In [49]:
y_pred_lstm_scaled = model_lstm_layers.predict(X_test)
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1502.87 MW
RMSE: 2076.88 MW
MAPE: 4.73%
R2: 0.90%


In [ ]:
#hyper parameter

In [50]:
input_shape = (X_train.shape[1], X_train.shape[2])

def build_model(hp):

    model = Sequential([
        Input(shape=input_shape),

        LSTM(
            units=hp.Choice(
                "lstm_units",
                [32, 64, 128]
            )
        ),

        BatchNormalization(),

        Dense(
            units=hp.Choice(
                "dense_units",
                [16, 32, 64]
            ),
            activation="relu"
        ),

        Dropout(
            hp.Choice(
                "dropout",
                [0.2, 0.3, 0.5]
            )
        ),

        Dense(HORIZON)
    ])

    model.compile(
         optimizer=hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd"]),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [51]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    directory="enn_tuning",
    project_name="lstm_forecasting"
)

In [52]:
tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Trial 5 Complete [00h 02m 06s]
val_loss: 0.010185007005929947

Best val_loss So Far: 0.001990286633372307
Total elapsed time: 00h 12m 57s


In [54]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("RNN units:", best_hp.get("lstm_units"))
print("Dense units:", best_hp.get("dense_units"))
print("Dropout:", best_hp.get("dropout"))

RNN units: 32
Dense units: 64
Dropout: 0.3


In [55]:
best_model = tuner.get_best_models(num_models=1)[0]

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 11 variables. 
  saveable.load_own_variables(store)


In [56]:
history_rnn = best_model.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 0.0029 - mae: 0.0409 - val_loss: 0.0020 - val_mae: 0.0342
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0029 - mae: 0.0408 - val_loss: 0.0019 - val_mae: 0.0322
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - mae: 0.0405 - val_loss: 0.0020 - val_mae: 0.0338
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - mae: 0.0405 - val_loss: 0.0019 - val_mae: 0.0322
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - mae: 0.0402 - val_loss: 0.0022 - val_mae: 0.0341
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - mae: 0.0402 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0028 - mae: 0.0402 - val_loss: 0.0024 - val_mae: 0.0358
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - mae: 0.0399 - val_loss: 0.0021 - val_mae: 0.0333
Epoch 9/10
1588/1588 ━━━

In [57]:
y_pred_bilstm_scaled = best_model.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1661.34 MW
RMSE: 2180.08 MW
MAPE: 5.43%
R2:   0.8855
